# Chapter 8 — Caveat Emptor
### Four ways a correctly computed sensitivity index can still mislead you

**Companion text:** L. M. Arriola and J. M. Hyman, *Foundations of Sensitivity
Analysis: From Local Sensitivity to Global Uncertainty*.
Manuscript in preparation for submission to SIAM (2026).
Not submitted, not under review, not accepted for publication.

**Sections covered:** §8.2 (bifurcations), §8.3 (ill-conditioning),
§8.5 (when global SA is needed), §8.7 (computational laboratory)
**Equations implemented:** the fold $f(u;p)=u^2+p$; the SIS endemic
equilibrium $i^\\ast = 1-1/\\mathcal{R}_0$; the conditioning bound
$\\|\\delta u\\|/\\|u\\| \\le \\kappa(A)\\,\\|\\delta b\\|/\\|b\\|$

**Learning objectives:**
- Separate the two ways a bifurcation defeats local SA: at a fold the *raw
  derivative* diverges; at a transcritical bifurcation the raw derivative stays
  finite and the *normalized index* diverges
- Watch $\\kappa_2$ of the SIR endemic Jacobian spike as $\\mathcal{R}_0 \\to 1^+$
- See that the conditioning bound is worst-case, and that a typical perturbation
  falls far short of it

**Prerequisites:** Chapters 2, 3 and 5.

**The chapter's point, and the book's.** Every index in this notebook is computed
correctly. Each one still misleads, because a sensitivity index is a statement
about a linearization of a model — not about the world, and not about the model
far from the point where the derivative was taken.


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

FULL = False
SEED = 20260724
rng  = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)
print("Chapter 8: Caveat Emptor")
print(f"FULL = {FULL}   seed = {SEED}")

In [ ]:
# ── §8.2  The fold: f(u; p) = u^2 + p ────────────────────────────────────────
# For p < 0 there are two real roots, u = +/- sqrt(-p); they merge at p = 0.
# The FSE gives  du*/dp = -1/(2 sqrt(-p)),  which diverges as p -> 0^-.
# The NORMALIZED index, however, is constant:
#     S^{u*}_p = (p/u*) du*/dp = 1/2   for every p < 0.
# Two different diagnostics, two different answers. That is the lesson.

def u_star(p):     return np.sqrt(-p)
def du_dp(p):      return -1.0/(2.0*np.sqrt(-p))
def SI_p_ustar(p): return (p/u_star(p))*du_dp(p)

def du_dp_fd(p, h):
    """Centered difference, as in §8.7 Part 1."""
    return (u_star(p + h) - u_star(p - h))/(2*h)

print(f"{'p':>10} {'u*':>12} {'du*/dp':>14} {'S^u*_p':>10}")
for p in (-4.0, -1.0, -0.1, -0.01, -0.001):
    print(f"{p:>10.4f} {u_star(p):>12.6f} {du_dp(p):>14.4f} {SI_p_ustar(p):>10.6f}")

In [ ]:
# ── §8.2  The transcritical bifurcation: the SIS endemic equilibrium ─────────
# i' = i(beta(1-i) - gamma_R), equilibria i* = 0 and i* = 1 - 1/R0 for R0 > 1.
# Here the RAW derivative stays finite; what diverges is the normalized index,
# because the quantity it is normalized against, i*, is itself going to zero:
#     S^{i*}_beta = 1/(R0 - 1).

def i_star(R0):        return 1.0 - 1.0/R0 if R0 > 1.0 else 0.0
def SI_beta_istar(R0): return 1.0/(R0 - 1.0)

def di_star_dbeta(beta, gamma_R):
    """Raw derivative d i*/d beta = gamma_R / beta^2 -- finite at R0 = 1."""
    return gamma_R/beta**2

print(f"{'R0':>7} {'i*':>12} {'raw di*/dbeta':>16} {'S^i*_beta':>12}")
GAMMA = 1.0/7.0
for R0 in (3.0, 2.0, 1.5, 1.2, 1.1, 1.01):
    beta = R0*GAMMA
    print(f"{R0:>7.2f} {i_star(R0):>12.6f} {di_star_dbeta(beta, GAMMA):>16.4f} "
          f"{SI_beta_istar(R0):>12.4f}")

In [ ]:
# ── Verification Suite ───────────────────────────────────────────────────────
print("=" * 68)
print("VERIFICATION SUITE  —  Chapter 8")
print("=" * 68)

# Test 1 -- the fold's normalized index is exactly 1/2, independent of p (§8.2)
for p in (-4.0, -1.0, -0.1, -0.01, -0.001, -1e-8):
    assert abs(SI_p_ustar(p) - 0.5) < 1e-12, \
        f"FAIL: S^u*_p = {SI_p_ustar(p)} at p = {p}, closed form gives 1/2"
print("Test 1 PASS  S^u*_p = 1/2 exactly, for every p < 0 tested (§8.2)")

# Test 2 -- but the raw derivative diverges: this is what the SI hides
d_far, d_near = abs(du_dp(-4.0)), abs(du_dp(-1e-6))
assert d_near > 1e3*d_far, "FAIL: du*/dp must blow up as p -> 0^-"
print(f"Test 2 PASS  |du*/dp| grows {d_near/d_far:.3e}x from p = -4 to p = -1e-6, "
      "while the SI never moves")

# Test 3 -- SIS: the closed form S^{i*}_beta = 1/(R0-1), checked against a
# centered difference on the equilibrium itself.
for R0 in (3.0, 2.0, 1.5, 1.2):
    beta, h = R0*GAMMA, 1e-7
    ip = 1.0 - GAMMA/(beta + h); im = 1.0 - GAMMA/(beta - h)
    si_fd = (beta/i_star(R0))*(ip - im)/(2*h)
    assert abs(si_fd - SI_beta_istar(R0)) < 1e-5, \
        f"FAIL: at R0 = {R0}, FD gives {si_fd}, closed form {SI_beta_istar(R0)}"
print("Test 3 PASS  S^i*_beta = 1/(R0-1) confirmed by centered differences")

# Test 4 -- the index diverges as R0 -> 1^+ while the raw derivative does not
assert SI_beta_istar(1.01) > SI_beta_istar(1.5) > SI_beta_istar(3.0), \
    "FAIL: the normalized index must grow without bound as R0 -> 1"
raw_ratio = di_star_dbeta(1.01*GAMMA, GAMMA)/di_star_dbeta(3.0*GAMMA, GAMMA)
assert raw_ratio < 10.0, "FAIL: the RAW derivative should stay bounded here"
print(f"Test 4 PASS  index goes {SI_beta_istar(3.0):.2f} -> {SI_beta_istar(1.01):.2f}, "
      f"while the raw derivative changes only {raw_ratio:.2f}x")

N_TESTS = 4
print(f"\nAll {N_TESTS} verification tests PASSED.")
print("=" * 68)

In [ ]:
# ── §8.3  Ill-conditioning ───────────────────────────────────────────────────
# The classical bound  ||du||/||u|| <= kappa_2(A) ||db||/||b||  is a WORST CASE.
# It is attained exactly, and it is also almost never what you observe.

def conditioning_worst_case(k, eps=1e-8):
    """A = diag(1, 1/k) has kappa_2 = k.

    The bound is attained when b lies along the direction A^{-1} stretches
    least and db along the direction it stretches most: b = e1, db = eps e2.
    """
    A  = np.diag([1.0, 1.0/k])
    b  = np.array([1.0, 0.0]); db = np.array([0.0, eps])
    u  = np.linalg.solve(A, b)
    du = np.linalg.solve(A, b + db) - u
    return (np.linalg.norm(du)/np.linalg.norm(u))/(np.linalg.norm(db)/np.linalg.norm(b))

def conditioning_typical(k, rng, n_trials=400, eps=1e-8):
    """The same matrix, with random b and db: the median amplification."""
    A, acc = np.diag([1.0, 1.0/k]), []
    for _ in range(n_trials):
        b  = rng.standard_normal(2); b /= np.linalg.norm(b)
        db = rng.standard_normal(2); db *= eps/np.linalg.norm(db)
        u  = np.linalg.solve(A, b)
        du = np.linalg.solve(A, b + db) - u
        acc.append((np.linalg.norm(du)/np.linalg.norm(u))/
                   (np.linalg.norm(db)/np.linalg.norm(b)))
    return float(np.median(acc))

kappas = np.array([1e1, 1e2, 1e3, 1e4, 1e5, 1e6])
worst  = np.array([conditioning_worst_case(k) for k in kappas])
typ    = np.array([conditioning_typical(k, rng) for k in kappas])
print(f"{'kappa':>10} {'worst case':>14} {'/kappa':>10} {'typical':>12}")
for k, w, t in zip(kappas, worst, typ):
    print(f"{k:>10.0e} {w:>14.4e} {w/k:>10.6f} {t:>12.4f}")

In [ ]:
# ── §8.7 Part 4  Condition number of the SIR endemic Jacobian vs R0 ─────────
# SIR with demography: S' = nu - c beta S I - nu S, etc.  The endemic
# equilibrium exists for R0 > 1 and the Jacobian degenerates as R0 -> 1^+.

def sir_endemic(c, beta, tau_R, tau_m):
    """Endemic equilibrium; R0 = c beta/(gamma_R + nu_m)."""
    g, nu = 1.0/tau_R, 1.0/tau_m
    R0 = c*beta/(g + nu)
    return R0, 1.0/R0, (nu/(g + nu))*(1.0 - 1.0/R0)

def sir_jacobian(c, beta, tau_R, tau_m):
    g, nu = 1.0/tau_R, 1.0/tau_m
    _, S, I = sir_endemic(c, beta, tau_R, tau_m)
    b = c*beta
    return np.array([[-b*I - nu, -b*S,          0.0],
                     [ b*I,       b*S - g - nu, 0.0],
                     [ 0.0,       g,           -nu]])

def beta_for_R0(R0, c, tau_R, tau_m):
    g, nu = 1.0/tau_R, 1.0/tau_m
    return R0*(g + nu)/c

C, TAUR, TAUM = 5.0, 7.0, 10000.0
R0s = np.array([1.01, 1.05, 1.1, 1.5, 2.0, 3.0])
kap = []
print(f"{'R0':>7} {'S*':>10} {'I*':>12} {'kappa_2(J_F)':>16}")
for R0 in R0s:
    b = beta_for_R0(R0, C, TAUR, TAUM)
    _, S, I = sir_endemic(C, b, TAUR, TAUM)
    k = np.linalg.cond(sir_jacobian(C, b, TAUR, TAUM), 2)
    kap.append(k)
    print(f"{R0:>7.2f} {S:>10.4f} {I:>12.4e} {k:>16.4e}")
kap = np.array(kap)

In [ ]:
# ── Verification Suite, part 2 ───────────────────────────────────────────────
print("=" * 68)
print("VERIFICATION SUITE  —  §8.3 and §8.7")
print("=" * 68)

# Test 5 -- the conditioning bound is attained EXACTLY by the extremal choice
for k, w in zip(kappas, worst):
    assert abs(w/k - 1.0) < 1e-8, f"FAIL: at kappa = {k:.0e} amplification/kappa = {w/k}"
slope = np.polyfit(np.log10(kappas), np.log10(worst), 1)[0]
assert abs(slope - 1.0) < 1e-8, f"FAIL: log-log slope {slope}, bound predicts exactly 1"
print(f"Test 5 PASS  worst-case amplification equals kappa exactly; "
      f"log-log slope = {slope:.10f}")

# Test 6 -- and a typical perturbation achieves almost none of it.  This is why
# a well-behaved-looking computation can still be one direction away from ruin.
assert np.max(typ) < 10.0, "FAIL: typical amplification should stay near 1"
print(f"Test 6 PASS  median amplification stays near {np.median(typ):.2f} "
      f"even at kappa = 1e6 -- the bound is worst case, not typical")

# Test 7 -- kappa_2 of the SIR endemic Jacobian rises monotonically as R0 -> 1^+
assert np.all(np.diff(kap) < 0), "FAIL: kappa_2 must decrease as R0 increases"
assert kap[0] > 10*kap[-1], "FAIL: expected a pronounced spike near R0 = 1"
print(f"Test 7 PASS  kappa_2 falls monotonically, {kap[0]:.2e} at R0 = 1.01 "
      f"down to {kap[-1]:.2e} at R0 = 3")

# Test 8 -- the endemic equilibrium collapses to the disease-free state at R0 = 1
_, _, I_near = sir_endemic(C, beta_for_R0(1.000001, C, TAUR, TAUM), TAUR, TAUM)
assert abs(I_near) < 1e-8, f"FAIL: I* = {I_near} should vanish as R0 -> 1^+"
print(f"Test 8 PASS  I* -> 0 as R0 -> 1^+  (I* = {I_near:.3e} at R0 = 1.000001)")

print(f"\nAll 4 tests PASSED  (8 in the notebook overall).")
print("=" * 68)

In [ ]:
# ── §8.7 Part 5  Sigma-normalized indices: when a tie is not a tie ──────────
# R0 = c beta tau_R is a monomial, so every local index is 1 -- the ranking is a
# three-way tie and tells you nothing about which parameter to measure better.
# Weighting by realistic 1-sigma uncertainty breaks the tie.

SIGMAS = dict(c=1.0, beta=0.02, tau_R=3.0, tau_m=500.0)   # §8.7 Part 5
NOM    = dict(c=5.0, beta=0.06, tau_R=7.0, tau_m=10000.0)

def local_SI_R0():
    """Monomial rule: R0 = c*beta*tau_R has unit elasticity in each factor.

    Unit elasticity, not 'linearity': a one-percent change in any factor gives a
    one-percent change in R0, which is a statement about proportions.
    """
    return dict(c=1.0, beta=1.0, tau_R=1.0, tau_m=0.0)

def sigma_SI_R0(nom, sig):
    """S^sigma_p = (sigma_p / R0) dR0/dp, with dR0/dp = R0/p for each factor."""
    return {p: (sig[p]/nom[p] if p in ("c", "beta", "tau_R") else 0.0) for p in nom}

loc, sg = local_SI_R0(), sigma_SI_R0(NOM, SIGMAS)
print(f"{'parameter':>10} {'local SI':>10} {'sigma-normalized':>20}")
for p in ("c", "beta", "tau_R", "tau_m"):
    print(f"{p:>10} {loc[p]:>10.3f} {sg[p]:>20.4f}")

order = sorted(("c", "beta", "tau_R"), key=lambda p: -sg[p])
assert len(set(loc[p] for p in ("c", "beta", "tau_R"))) == 1, \
    "FAIL: the local indices should tie at 1"
assert order == ["tau_R", "beta", "c"], f"FAIL: sigma ranking {order}"
print(f"\nCheck PASS  local indices tie at 1; sigma-normalized ranking is "
      f"{' > '.join(order)}")
print("The tie was never informative. The uncertainty ranges carry the decision.")

In [ ]:
# ── Figures ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ps = np.linspace(-4.0, -0.005, 400)
axes[0].plot(ps, u_star(ps), lw=2, label="$u^*=\\sqrt{-p}$")
ax0b = axes[0].twinx()
ax0b.plot(ps, np.abs(du_dp(ps)), color="coral", ls="--", lw=2, label="$|du^*/dp|$")
ax0b.set_yscale("log"); ax0b.set_ylabel("$|du^*/dp|$ (log)", color="coral")
axes[0].set_xlabel("$p$"); axes[0].set_ylabel("$u^*$")
axes[0].set_title("Fold: root is smooth,\nderivative is not")
axes[0].grid(alpha=0.3)

rr = np.linspace(1.005, 3.0, 400)
axes[1].plot(rr, [i_star(r) for r in rr], lw=2, label="$i^*=1-1/\\mathcal{R}_0$")
ax1b = axes[1].twinx()
ax1b.plot(rr, [SI_beta_istar(r) for r in rr], color="coral", ls="--", lw=2)
ax1b.set_yscale("log"); ax1b.set_ylabel("$S^{i^*}_\\beta$ (log)", color="coral")
axes[1].set_xlabel("$\\mathcal{R}_0$"); axes[1].set_ylabel("$i^*$")
axes[1].set_title("Transcritical: equilibrium is smooth,\nnormalized index diverges")
axes[1].grid(alpha=0.3)

axes[2].loglog(kappas, worst, "o-", lw=2, label="worst case (attains the bound)")
axes[2].loglog(kappas, typ, "s--", lw=2, label="typical (median of 400 draws)")
axes[2].set_xlabel("$\\kappa_2(A)$"); axes[2].set_ylabel("error amplification")
axes[2].set_title("The conditioning bound is\nworst case, not typical")
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3, which="both")

plt.tight_layout(); plt.savefig("ch08_fig_failure_modes.pdf", bbox_inches="tight")
plt.show()
print("Figure saved: ch08_fig_failure_modes.pdf")

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────────
print("=" * 68)
print("CHAPTER 8 SUMMARY  —  four correct indices, four ways to be misled")
print("=" * 68)
print(f"""
1. Fold, f(u;p) = u^2 + p
   S^u*_p = 1/2 for every p < 0 -- perfectly stable, and useless as a warning.
   The raw derivative meanwhile runs from {abs(du_dp(-4.0)):.3f} to {abs(du_dp(-1e-6)):.3e}.
   Reporting only the normalized index hides the fold completely.

2. Transcritical, SIS endemic equilibrium
   The opposite failure. The raw derivative stays finite; the normalized index
   goes {SI_beta_istar(3.0):.2f} -> {SI_beta_istar(1.5):.2f} -> {SI_beta_istar(1.01):.2f}
   as R0 -> 1, because i* itself is going to zero.

3. Ill-conditioning
   kappa_2 of the SIR endemic Jacobian: {kap[0]:.2e} at R0 = 1.01, {kap[-1]:.2e} at R0 = 3.
   The bound ||du||/||u|| <= kappa ||db||/||b|| is attained EXACTLY in the worst
   direction, and a typical perturbation sees an amplification near {np.median(typ):.1f}.
   A computation that looks fine is not evidence that it is.

4. A ranking that is not a ranking
   The local indices for R0 = c beta tau_R all equal 1 -- a three-way tie by
   construction. Weighted by realistic uncertainty the order is
   tau_R ({sg['tau_R']:.3f}) > beta ({sg['beta']:.3f}) > c ({sg['c']:.3f}).

Every number above is correct. Each becomes misleading the moment it is read as
a statement about the world rather than about a linearization of a model.
""")
print("=" * 68)

In [ ]:
# ── Download (always the last cell) ──────────────────────────────────────────
try:
    from google.colab import files
    files.download("ch08_fig_failure_modes.pdf")
except ImportError:
    print("Not in Colab — figure saved locally: ch08_fig_failure_modes.pdf")